# Библиотеки

In [2]:
from itertools import combinations
from pprint import pprint

import pandas as pd
import numpy as np
from scipy import stats
import pingouin as pg
import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split # разбивка на тестовую и обучающую выборку
from sklearn.metrics import classification_report, precision_recall_curve, roc_curve, roc_auc_score # классификационный отчет с метриками качества, а также данные под кривые для визуалок + мера площади под кривой

import seaborn as sns

## 11.5 Мультиномиальная логистическая регрессия

Это те же данные по подпискам, только с добавленной еще одной группой (закодирована значением 1) пользователей, которые перешли на ограниченную по функционалу бета-версию программы. Т.о., столбец "Группа" содержит 3 значения\категории\группы: 

- 0 - отказались
- 1 -  подписались на бета-версию программы
- 2 - оформил полную платную подписку

In [3]:
d = pd.read_excel('./data/WebService3.xlsx')

In [7]:
d.head()

,Группа,ДоходМес,ОценкаИнтернет,ВремяПрилож
0,0,48.5,5,4
1,0,55.5,4,5
2,0,50.0,6,3
3,0,58.5,5,7
4,0,61.0,3,6


In [4]:
model = smf.mnlogit('Группа ~ ДоходМес + ОценкаИнтернет + ВремяПрилож', data=d)
result = model.fit()

Optimization terminated successfully.
         Current function value: 0.152160
         Iterations 10


In [5]:
result.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                          MNLogit Regression Results                          
==============================================================================
Dep. Variable:                 Группа   No. Observations:                   89
Model:                        MNLogit   Df Residuals:                       81
Method:                           MLE   Df Model:                            6
Date:                Mon, 05 Jan 2026   Pseudo R-squ.:                  0.8614
Time:                        21:29:18   Log-Likelihood:                -13.542
converged:                       True   LL-Null:                       -97.697
Covariance Type:            nonrobust   LLR p-value:                 1.026e-33
==================================================================================
      Группа=1       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         -9.6952      6.071     -1.597      0.110     -21.594       2.203
ДоходМес          -0.2248      0.119     -1.883      0.060      -0.459       0.009
ОценкаИнтернет     2.1032      0.784      2.683      0.007       0.567       3.640
ВремяПрилож        0.7422      0.234      3.177      0.001       0.284       1.200
----------------------------------------------------------------------------------
      Группа=2       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept        -28.8390      9.915     -2.909      0.004     -48.273      -9.405
ДоходМес           0.1825      0.076      2.391      0.017       0.033       0.332
ОценкаИнтернет     1.8166      0.708      2.566      0.010       0.429       3.204
ВремяПрилож        0.4504      0.191      2.363      0.018       0.077       0.824
==================================================================================
"""

In [9]:
d['pred'] = result.predict(d).idxmax(axis=1)

- `precision` — если модель сказала, что объект относится к этому классу, насколько часто она права. Про доверие к предсказанию.
- `recall` — из всех реальных объектов этого класса, какую долю модель вообще смогла найти. Про полноту находок.
- `f1-score` — компромисс между precision и recall, одна цифра, чтобы не выбирать между ними.

Пример: модель нашла 10 подписчиков, 9 угадала, precision высокий. Но если всего подписчиков было 100, recall будет плохой.

In [11]:
print(classification_report(d['Группа'], d['pred']))

              precision    recall  f1-score   support

           0       0.97      0.90      0.93        31
           1       0.97      0.97      0.97        30
           2       0.90      0.96      0.93        28

    accuracy                           0.94        89
   macro avg       0.94      0.94      0.94        89
weighted avg       0.95      0.94      0.94        89



In [10]:
pd.crosstab(d['Группа'], d['pred'])

pred,0,1,2
Группа,,,
0,28,1,2
1,0,29,1
2,1,0,27
